# Capítulo 3: Análisis de Datos con Spark y MapReduce — Laboratorio práctico
**Tópicos Avanzados para Data Science en Python**

En este notebook usaremos **PySpark** dentro de Google Colab:

| Tema | Herramientas | Diapositivas |
|---|---|---|
| Sesión de Spark | `SparkSession`, `SparkContext` | 46-48 |
| RDDs y particiones | `parallelize`, `getNumPartitions`, `glom` | 45 |
| Transformaciones y acciones | `map`, `filter`, `collect`, `count`, *lazy evaluation* | 49 |
| MapReduce | `flatMap`, `map`, `reduceByKey` | 52-54 |
| Shuffle y stages | `groupByKey` vs `reduceByKey`, `toDebugString` | 45, 54 |
| DataFrames | `read.csv`, `select`, `filter`, `groupBy`, `explain` | 50-51 |
| Spark SQL | `createOrReplaceTempView`, `spark.sql` | 51 |
| Broadcast y Accumulator | `sc.broadcast`, `sc.accumulator` | 45 |
| Tolerancia a fallos | *lineage*, `cache`, `checkpoint` | 55 |
| Escribir resultados | `write.csv`, `write.parquet`, `partitionBy` | 54 |

### ¿Dónde está el clúster?
En Colab Spark corre en **modo local** (`local[*]`): el *driver* y los *executors* viven en la misma máquina y cada núcleo hace de "nodo". **El código es exactamente el mismo** que se ejecuta en un clúster real (Databricks, EMR, Dataproc); solo cambia dónde corre.

```
          Driver (tu notebook)
                 │  reparte tareas
     ┌───────────┼───────────┐
  Executor    Executor    Executor     ← en Colab: núcleos de la misma máquina
  [part 0]    [part 1]    [part 2]     ← cada uno procesa sus particiones
```

## 1. Preparación del laboratorio
Spark está escrito en Scala y corre sobre la **JVM**, por eso necesita **Java**. PySpark es la librería de Python que se comunica con esa JVM.

| Paso | ¿Qué hace? |
|---|---|
| Instalar Java y PySpark | Colab ya trae Java; instalamos PySpark 3.5 (compatible con Java 8, 11 y 17) |
| `SparkSession.builder...getOrCreate()` | Crea el **punto de entrada** a Spark (DataFrames y SQL) |
| `spark.sparkContext` | El **SparkContext** dentro de la sesión (RDDs) |
| `master("local[*]")` | Usa todos los núcleos de la máquina como *executors* |

In [ ]:
# Java + PySpark (tarda ~1 minuto la primera vez)
import shutil
if shutil.which("java") is None:
    !apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!java -version 2>&1 | head -1
!pip -q install "pyspark==3.5.*"

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("BSG-Capitulo3")
         .config("spark.ui.showConsoleProgress", "false")   # sin barras de progreso
         .getOrCreate())
sc = spark.sparkContext          # el SparkContext vive dentro de la SparkSession
sc.setLogLevel("FATAL")          # oculta los logs de la JVM (los errores igual llegan a Python)

print("Versión de Spark:", spark.version)

### Datos de práctica
La siguiente celda crea la carpeta `/content/spark_lab/` con dos archivos (puedes volver a ejecutarla para **reiniciar** el laboratorio):

```
/content/spark_lab/
├── datos/
│   ├── ventas.csv     ← 303 000 ventas "sucias" (las mismas reglas que src/extraer.py del Capítulo 2)
│   └── quijote.txt    ← texto para el clásico conteo de palabras
└── salida/            ← aquí guardaremos resultados
```

Las ventas traen errores a propósito: regiones mal escritas (`LIMA`, `  cusco `), cantidades vacías, precios negativos y ~1% de registros duplicados.

In [ ]:
import csv, os, random
from datetime import date, timedelta

LAB = "/content/spark_lab"
DATOS = f"{LAB}/datos"
SALIDA = f"{LAB}/salida"
RUTA_VENTAS = f"{DATOS}/ventas.csv"
RUTA_TEXTO = f"{DATOS}/quijote.txt"

shutil.rmtree(LAB, ignore_errors=True)
os.makedirs(DATOS)
os.makedirs(SALIDA)

# ---- ventas.csv -----------------------------------------------------------
rng = random.Random(42)
REGIONES = ["Lima", "Arequipa", "Cusco", "Piura", "Trujillo"]
PRODUCTOS = {"Laptop": 3500.0, "Monitor": 850.0, "Teclado": 120.0,
             "Mouse": 60.0, "Audifonos": 250.0}
# Cada región prefiere productos distintos (pesos en el orden de PRODUCTOS)
PREFERENCIAS = {"Lima": [30, 25, 15, 20, 10], "Arequipa": [10, 15, 30, 35, 10],
                "Cusco": [10, 15, 15, 25, 35], "Piura": [15, 28, 32, 15, 10],
                "Trujillo": [12, 34, 14, 12, 28]}
N = 300_000

filas = []
for i in range(1, N + 1):
    region = rng.choice(REGIONES)
    producto = rng.choices(list(PRODUCTOS), weights=PREFERENCIAS[region])[0]
    precio = round(PRODUCTOS[producto] * rng.uniform(0.9, 1.1), 2)
    cantidad = rng.randint(1, 10)
    r = rng.random()
    if r < 0.10:
        region = region.upper()
    elif r < 0.20:
        region = f"  {region.lower()} "
    if rng.random() < 0.02:
        cantidad = ""
    if rng.random() < 0.01:
        precio = -precio
    fecha = (date(2026, 1, 1) + timedelta(days=rng.randrange(180))).isoformat()
    filas.append([i, fecha, region, producto, cantidad, precio])
filas += rng.sample(filas, N // 100)          # duplicados del sistema de origen

with open(RUTA_VENTAS, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id_venta", "fecha", "region", "producto", "cantidad", "precio_unitario"])
    w.writerows(filas)

# ---- quijote.txt ----------------------------------------------------------
TEXTO = '''En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo
que vivía un hidalgo de los de lanza en astillero, adarga antigua, rocín flaco y galgo corredor.
Una olla de algo más vaca que carnero, salpicón las más noches, duelos y quebrantos los sábados,
lentejas los viernes, algún palomino de añadidura los domingos, consumían las tres partes de su hacienda.
El resto della concluían sayo de velarte, calzas de velludo para las fiestas,
con sus pantuflos de lo mismo, y los días de entre semana se honraba con su vellorí de lo más fino.
Tenía en su casa una ama que pasaba de los cuarenta, y una sobrina que no llegaba a los veinte,
y un mozo de campo y plaza, que así ensillaba el rocín como tomaba la podadera.
Frisaba la edad de nuestro hidalgo con los cincuenta años; era de complexión recia,
seco de carnes, enjuto de rostro, gran madrugador y amigo de la caza.'''
with open(RUTA_TEXTO, "w", encoding="utf-8") as f:
    f.write(TEXTO + "\n")

print(f"{len(filas):,} ventas en {RUTA_VENTAS}")
!head -5 {RUTA_VENTAS}
!wc -l {RUTA_TEXTO}

### ✏️ Ejercicio 1.1 — Conociendo tu sesión de Spark
1. Muestra la versión de Spark, el `master` (`sc.master`) y el nombre de la aplicación (`sc.appName`).
2. Muestra `sc.defaultParallelism`: el número de tareas que Spark ejecuta en paralelo por defecto.
3. Compáralo con el número de núcleos de la máquina usando el comando `nproc` del Capítulo 1.
4. Responde en un comentario: en esta sesión, ¿quién es el *driver* y quiénes son los *executors*?

In [ ]:
# Tu código aquí

## 2. RDDs y particiones
Un **RDD** (*Resilient Distributed Dataset*) es una colección **inmutable** repartida en **particiones**. Cada partición la procesa una *task* en un *executor*.

| Método | ¿Qué hace? |
|---|---|
| `sc.parallelize(lista, n)` | Crea un RDD a partir de una lista de Python, en `n` particiones |
| `sc.textFile(ruta)` | Crea un RDD donde cada elemento es una **línea** del archivo |
| `rdd.getNumPartitions()` | Número de particiones |
| `rdd.glom()` | Agrupa cada partición en una lista (para **ver** cómo se repartieron los datos) |
| `rdd.repartition(n)` | Redistribuye los datos en `n` particiones (provoca *shuffle*) |

In [ ]:
numeros = sc.parallelize(range(1, 21), 4)
print("Particiones:", numeros.getNumPartitions())
print("Contenido de cada partición:", numeros.glom().collect())

In [ ]:
# Un archivo de texto también se reparte: cada elemento del RDD es una línea
lineas = sc.textFile(RUTA_VENTAS)
print("Particiones:", lineas.getNumPartitions())
print("Primeras líneas:", lineas.take(3))

### ✏️ Ejercicio 2.1 — Repartir el trabajo
1. Crea un RDD con los números del 1 al 100 repartidos en **5** particiones.
2. Muestra cuántos elementos tiene cada partición (pista: `glom().map(len)`).
3. Calcula la suma total con la acción `reduce`.
4. Cambia a **2** particiones con `repartition` y vuelve a mostrar cuántos elementos tiene cada una.

In [ ]:
# Tu código aquí

## 3. Transformaciones, acciones y *lazy evaluation*
Las **transformaciones** solo **anotan** qué hacer y devuelven un RDD nuevo. Las **acciones** disparan la ejecución (crean un *job*) y devuelven un resultado al *driver*.

| Transformaciones (perezosas) | Acciones (ejecutan) |
|---|---|
| `map`, `filter`, `flatMap` | `collect`, `take(n)`, `first` |
| `distinct`, `sortBy` | `count`, `reduce` |
| `reduceByKey`, `groupByKey`, `join` | `saveAsTextFile`, `foreach` |

> ⚠️ `collect()` trae **todo** el RDD a la memoria del *driver*. Con datos grandes usa `take(n)`.

In [ ]:
import time

inicio = time.time()
grande = sc.parallelize(range(10_000_000), 8)
pares = grande.filter(lambda x: x % 2 == 0).map(lambda x: x * 10)   # transformaciones
print(f"Transformaciones: {time.time() - inicio:.4f} s  ← no se ejecutó nada")

inicio = time.time()
print("Resultado:", pares.count())                                   # acción
print(f"Acción: {time.time() - inicio:.2f} s  ← aquí recién trabaja Spark")

In [ ]:
# El *lineage*: la "receta" que Spark guarda para poder recalcular cualquier partición
print(pares.toDebugString().decode())

### ✏️ Ejercicio 3.1 — ¿Cuándo se ejecuta el código?
1. Crea un RDD con `[5, 0, 2]` y aplícale `map(lambda x: 100 / x)`. ¿Falla esa línea?
2. Llama a la acción `collect()` dentro de un `try/except` e imprime un mensaje si falla.
3. Corrige el RDD filtrando los ceros **antes** del `map` y muestra el resultado.
4. Responde en un comentario: ¿por qué el error no aparece en el paso 1?

In [ ]:
# Tu código aquí

## 4. MapReduce con RDDs
| Fase MapReduce | En PySpark | Ejemplo: "el gato y el perro" |
|---|---|---|
| Entrada | `sc.textFile` / `parallelize` | `"el gato y el perro"` |
| **Map** | `flatMap` (1 → muchos) y `map` (1 → 1) | `(el,1) (gato,1) (y,1) (el,1) (perro,1)` |
| **Shuffle & Sort** | automático | `el → [1,1]`, `gato → [1]`, ... |
| **Reduce** | `reduceByKey(func)` | `(el,2) (gato,1) (y,1) (perro,1)` |

`reduceByKey` trabaja con **pares clave-valor**: tuplas `(clave, valor)`.

In [ ]:
frase = sc.parallelize(["el gato y el perro"])
palabras = frase.flatMap(lambda linea: linea.split())           # Map: separar palabras
pares = palabras.map(lambda palabra: (palabra, 1))              # Map: (clave, valor)
conteo = pares.reduceByKey(lambda a, b: a + b)                  # Shuffle + Reduce
print("Map:   ", pares.collect())
print("Reduce:", conteo.collect())

In [ ]:
# El mismo conteo sobre el archivo, ordenado de mayor a menor
conteo_texto = (sc.textFile(RUTA_TEXTO)
                .flatMap(lambda linea: linea.split())
                .map(lambda palabra: (palabra, 1))
                .reduceByKey(lambda a, b: a + b))
conteo_texto.takeOrdered(5, key=lambda par: -par[1])

### ✏️ Ejercicio 4.1 — Conteo de palabras limpio
El conteo anterior trata `"Mancha,"` y `"Mancha"` como palabras distintas, y `"En"` y `"en"` también.
1. Lee `RUTA_TEXTO` y pasa cada línea a minúsculas.
2. Extrae solo las palabras con `re.findall(r"[a-záéíóúñü]+", linea)`.
3. Elimina las palabras de la lista `STOPWORDS` (artículos, preposiciones...).
4. Muestra las **10** palabras más frecuentes con `takeOrdered`.

```python
STOPWORDS = {"de", "la", "que", "el", "en", "y", "a", "los", "se", "del", "las", "un", "por",
             "con", "no", "una", "su", "sus", "para", "es", "al", "lo", "como", "más", "o", "ha"}
```

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 4.2 — Total vendido por región con MapReduce
Usando **solo RDDs** sobre `RUTA_VENTAS`:
1. Lee el archivo con `sc.textFile` y quita la cabecera (pista: `filter` con la primera línea, `lineas.first()`).
2. Separa cada línea por comas.
3. Descarta las filas con `cantidad` vacía o `precio_unitario <= 0`.
4. **Map**: genera pares `(region, cantidad * precio)`, normalizando la región con `.strip().title()`.
5. **Reduce**: suma por región y muestra el resultado ordenado de mayor a menor, redondeado a 2 decimales.

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 4.3 — Precio promedio por producto
El promedio **no** se puede calcular directamente con `reduceByKey`: el promedio de promedios no es el promedio total.
El truco clásico de MapReduce es reducir **pares (suma, conteo)** y dividir al final.
1. Genera pares `(producto, (precio_unitario, 1))` solo con precios positivos (no hace falta filtrar la cantidad).
2. Con `reduceByKey` suma los precios y los conteos a la vez.
3. Con `mapValues` calcula `suma / conteo` y redondea a 2 decimales.
4. Responde en un comentario: ¿por qué la función de Reduce debe ser asociativa (diapositiva 53)?

In [ ]:
# Tu código aquí

## 5. Shuffle y stages
| Tipo de transformación | Ejemplos | ¿Mueve datos entre nodos? |
|---|---|---|
| **Estrecha** (*narrow*) | `map`, `filter`, `flatMap` | No: cada partición se procesa sola |
| **Amplia** (*wide*) | `reduceByKey`, `groupByKey`, `distinct`, `join`, `sortBy`, `repartition` | Sí: **shuffle** |

Cada shuffle **corta el plan en un nuevo stage**. En `toDebugString`, cada nivel de sangría es un stage y `ShuffledRDD` marca el corte.

`reduceByKey` combina los valores **dentro de cada partición antes** del shuffle (como un mini-Reduce local); `groupByKey` envía **todos** los valores por la red. Por eso `reduceByKey` es preferible.

In [ ]:
print(conteo_texto.toDebugString().decode())

### ✏️ Ejercicio 5.1 — groupByKey vs reduceByKey
1. A partir de `RUTA_VENTAS` (sin cabecera), genera pares `(producto, 1)`.
2. Cuenta las ventas por producto de dos formas: con `reduceByKey` y con `groupByKey().mapValues(len)`.
3. Comprueba que ambos resultados son iguales (ordénalos con `sorted`).
4. Imprime el `toDebugString` de la versión con `reduceByKey` y señala dónde está el shuffle.
5. Responde en un comentario: de la lista `map, filter, reduceByKey, distinct, flatMap, join, sortBy`, ¿cuáles provocan shuffle?

In [ ]:
# Tu código aquí

## 6. DataFrames
Un DataFrame es una tabla distribuida con **columnas y tipos**. Es la API recomendada: Spark optimiza el plan completo (optimizador **Catalyst**) antes de ejecutarlo.

| pandas | PySpark | ¿Qué hace? |
|---|---|---|
| `pd.read_csv(ruta)` | `spark.read.csv(ruta, header=True, inferSchema=True)` | Leer CSV |
| `df.dtypes` | `df.printSchema()` | Ver columnas y tipos |
| `df.head()` | `df.show(5)` | Ver filas (`show` es una acción) |
| `df[["a", "b"]]` | `df.select("a", "b")` | Elegir columnas |
| `df[df.a > 0]` | `df.filter(F.col("a") > 0)` | Filtrar filas |
| `df["c"] = ...` | `df.withColumn("c", ...)` | Crear o reemplazar una columna |
| `df.groupby("a").agg(...)` | `df.groupBy("a").agg(F.sum("b"))` | Agrupar |
| `df.sort_values("a")` | `df.orderBy(F.desc("a"))` | Ordenar |
| `df.drop_duplicates()` / `df.dropna()` | `df.dropDuplicates([...])` / `df.dropna(subset=[...])` | Limpiar |
| `str.strip().str.title()` | `F.initcap(F.trim(col))` | Normalizar texto |

In [ ]:
ventas = spark.read.csv(RUTA_VENTAS, header=True, inferSchema=True)
ventas.printSchema()
ventas.show(5)

In [ ]:
# Los problemas de calidad saltan a la vista al agrupar por región
ventas.groupBy("region").count().orderBy("region").show(20)

In [ ]:
# explain() muestra el plan físico. "Exchange" = shuffle (el groupBy necesita juntar cada región)
ventas.groupBy("region").count().explain()

### ✏️ Ejercicio 6.1 — Limpiar las ventas
Crea un DataFrame `df_limpio` a partir de `ventas` que:
1. Elimine duplicados por `id_venta`.
2. Elimine filas con `cantidad` nula y con `precio_unitario <= 0`.
3. Normalice `region` con `F.initcap(F.trim(...))`.
4. Agregue la columna `total = round(cantidad * precio_unitario, 2)` (pista: `F.round`).

Muestra cuántas filas había antes y después, y las regiones distintas que quedaron (deben ser 5).

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 6.2 — Resumen por región con DataFrames
Con `df_limpio`:
1. Calcula por región el número de ventas (`F.count`) y el total vendido (`F.sum`). Para que no salga en notación científica, conviértelo a decimal: `.cast("decimal(18,2)")`.
2. Ordena de mayor a menor total.
3. Compara con el resultado del Ejercicio 4.2. ¿Los totales coinciden? Explica en un comentario por qué.

In [ ]:
# Tu código aquí

## 7. Spark SQL
Cualquier DataFrame se puede consultar con SQL estándar registrándolo como **vista temporal**:

| Método | ¿Qué hace? |
|---|---|
| `df.createOrReplaceTempView("nombre")` | Registra el DataFrame como una tabla para SQL |
| `spark.sql("SELECT ...")` | Ejecuta la consulta y devuelve un **DataFrame** |

DataFrames y SQL generan **el mismo plan**: elige la sintaxis con la que el equipo se sienta más cómodo.

In [ ]:
ventas.createOrReplaceTempView("ventas_crudas")
spark.sql('''
    SELECT producto, COUNT(*) AS n_ventas
    FROM ventas_crudas
    GROUP BY producto
    ORDER BY n_ventas DESC
''').show()

### ✏️ Ejercicio 7.1 — Consultas con SQL
1. Registra `df_limpio` como la vista `ventas_limpias`.
2. Con SQL, obtén los **3 productos** con mayor total vendido en **Lima**.
3. Con SQL, obtén el total vendido **por mes** (pista: `date_format(fecha, 'yyyy-MM')`), ordenado por mes.

> 💡 Usa `CAST(SUM(total) AS DECIMAL(18,2))` para que los montos no salgan en notación científica.

In [ ]:
# Tu código aquí

## 8. Variables compartidas: Broadcast y Accumulator
| Variable | Dirección | Uso típico |
|---|---|---|
| **Broadcast** `sc.broadcast(valor)` | Driver → todos los executors (solo lectura) | Tablas pequeñas de referencia (diccionarios, catálogos) |
| **Accumulator** `sc.accumulator(0)` | Executors → driver (solo se suma) | Contadores: registros inválidos, errores |

Sin broadcast, Spark enviaría una copia del diccionario **con cada tarea**; con broadcast se envía **una vez por executor**.

> ⚠️ Actualiza los accumulators dentro de una **acción** (`foreach`). Dentro de una transformación, si Spark reintenta una tarea, el contador podría sumarse dos veces.

In [ ]:
zonas = sc.broadcast({"Lima": "Centro", "Arequipa": "Sur", "Cusco": "Sur",
                      "Piura": "Norte", "Trujillo": "Norte"})
print(zonas.value["Cusco"])

lineas_vacias = sc.accumulator(0)
sc.parallelize(["a", "", "b", "", ""]).foreach(lambda x: lineas_vacias.add(1) if x == "" else None)
print("Líneas vacías:", lineas_vacias.value)

### ✏️ Ejercicio 8.1 — Total por zona y control de calidad
1. Usando el broadcast `zonas` y RDDs sobre `RUTA_VENTAS`, calcula el total vendido (`cantidad * precio`) por **zona** (Centro, Sur, Norte), solo con filas válidas (cantidad no vacía y precio > 0).
2. Con un **accumulator** y `foreach`, cuenta cuántas filas (sin la cabecera) tienen la `cantidad` vacía.
3. Con otro accumulator, cuenta cuántas tienen `precio_unitario` negativo.

In [ ]:
# Tu código aquí

## 9. Tolerancia a fallos: *lineage*, `cache` y `checkpoint`
MapReduce (Hadoop) se protege de las fallas **escribiendo a disco** después de cada fase. Spark se protege con el **lineage**: si se pierde una partición, la **recalcula** siguiendo la receta.

| Método | ¿Qué hace? |
|---|---|
| `df.cache()` / `rdd.cache()` | Guarda el resultado en memoria la **primera vez** que una acción lo calcula |
| `df.unpersist()` | Libera esa memoria |
| `sc.setCheckpointDir(ruta)` + `rdd.checkpoint()` | Guarda el RDD en disco y **corta** el lineage (útil en cadenas muy largas o iterativas) |

Usa `cache` cuando vas a reutilizar el mismo resultado en **varias acciones**.

In [ ]:
sc.setCheckpointDir(f"{LAB}/checkpoints")

larga = sc.parallelize(range(1000), 4)
for _ in range(5):
    larga = larga.map(lambda x: x + 1)
print("Antes del checkpoint:\n", larga.toDebugString().decode())

larga.checkpoint()
larga.count()                      # la acción materializa el checkpoint
print("Después del checkpoint:\n", larga.toDebugString().decode())

### ✏️ Ejercicio 9.1 — ¿Cuánto ahorra el cache?
1. Lee de nuevo `RUTA_VENTAS` en un DataFrame y aplica la misma limpieza del 6.1 (llámalo `df_cache`).
2. Mide el tiempo de **dos** `count()` seguidos **sin** cache.
3. Aplica `df_cache.cache()` y vuelve a medir dos `count()` seguidos.
4. Libera la memoria con `unpersist()`.
5. Responde en un comentario: ¿cuál fue el más rápido y por qué el primer `count()` después de `cache()` no lo es?

In [ ]:
# Tu código aquí

## 10. Escribir resultados
Igual que los *Reduce workers* de la diapositiva 54, **cada partición escribe su propio archivo** (`part-0000x`). Por eso Spark guarda una **carpeta**, no un archivo.

| Método | ¿Qué hace? |
|---|---|
| `df.write.mode("overwrite").csv(ruta, header=True)` | Guarda en CSV (una carpeta con archivos `part-*`) |
| `df.write.parquet(ruta)` | Guarda en **Parquet**: columnar, comprimido y con tipos (el formato estándar en big data) |
| `df.write.partitionBy("col")` | Crea una subcarpeta por cada valor de la columna (`col=valor/`) |
| `df.coalesce(1)` | Junta todo en 1 partición → 1 archivo (solo para resultados pequeños) |
| `spark.read.parquet(ruta)` | Leer de vuelta |

In [ ]:
ventas.repartition(3).write.mode("overwrite").csv(f"{SALIDA}/ventas_copia", header=True)
!ls -lh {SALIDA}/ventas_copia

### ✏️ Ejercicio 10.1 — Guardar el trabajo
1. Guarda `df_limpio` repartido en **4** particiones como CSV en `SALIDA/ventas_limpias_csv` y lista la carpeta. ¿Cuántos archivos `part-` hay?
2. Guárdalo otra vez con `coalesce(1)` en `SALIDA/ventas_limpias_1archivo` y lista la carpeta.
3. Guarda `df_limpio` en Parquet **particionado por región** en `SALIDA/ventas_parquet` y muestra la estructura de carpetas con `find ... -type d` (Capítulo 1).
4. Lee de vuelta el Parquet y muestra el número de filas de **Cusco** (Spark solo leerá la carpeta `region=Cusco`).
5. Compara el tamaño en disco del CSV y del Parquet con `du -sh`.

In [ ]:
# Tu código aquí

## 11. 🏆 Reto integrador: top 2 productos por región
La gerencia quiere saber cuáles son los **2 productos con más unidades vendidas (suma de `cantidad`) en cada región**. Resuélvelo de **dos formas** a partir de `df_limpio` y comprueba que dan lo mismo:

**a) MapReduce con RDDs** (`df_limpio.rdd` devuelve un RDD de filas; accede a los campos con `fila.region`):
1. Map a `((region, producto), cantidad)` y Reduce sumando.
2. Map a `(region, (producto, unidades))`, agrupa por región y quédate con los 2 mayores.

**b) DataFrames**: agrupa por región y producto, y usa una **ventana** para numerar los productos dentro de cada región:
```python
from pyspark.sql.window import Window
ventana = Window.partitionBy("region").orderBy(F.desc("unidades"))
df.withColumn("puesto", F.row_number().over(ventana))
```

Finalmente:
3. Compara ambos resultados e imprime si son iguales.
4. Guarda el resultado de b) en Parquet en `SALIDA/top2_por_region`.
5. Responde en un comentario: ¿cuál de las dos versiones te resultó más fácil de leer? ¿cuántos shuffles crees que hace cada una?

In [ ]:
# Tu código aquí

---
## Fin de los ejercicios
Las soluciones están en el notebook **`Capitulo3_Soluciones.ipynb`**. Intenta resolver todo antes de revisarlas.

Para liberar la memoria de Colab al terminar:
```python
spark.stop()
```